# Previsão de Preços no E-commerce Brasileiro com Machine Learning: Base Olist

Machine Learning para Predição de Preços no E-commerce

Neste projeto, vou desenvolver um modelo de Machine Learning para estimar o preço de produtos em um marketplace. O projeto envolve aprendizado supervisionado para um problema de regressão, no qual a variável-alvo será o preço do produto.

Utilizarei a base pública da Olist, um marketplace brasileiro, contendo informações sobre produtos, vendedores, pedidos e outras características que podem ajudar a explicar as diferenças de preços.

Vou utilizar o seguinte pipeline:

1. Definição do problema de negócio.
2. Limpeza e pré-processamento dos dados.
3. Entendimento do problema e definição da estratégia de validação.
4. Divisão dos dados em conjuntos de treino e teste.
5. Exploração dos dados (EDA).
6. Treinamento dos modelos e ajuste de hiperparâmetros
7. Teste e avaliação final do modelo.
8. Interpretação dos resultados e geração de insights de negócio.


Neste notebook, realizaremos as etapas 6 e 8 do projeto, com foco na escolha dos melhores métodos e modelos para a variável alvo price.




Bibliotecas

Importamos as bibliotecas usadas no carregamento, manipulação, visualização e modelagem dos dados.

In [0]:
%pip install catboost==1.2.10 xgboost==3.4.1 -q
%pip install seaborn 

In [0]:
from pyspark.sql import SparkSession
spark = SparkSession.builder.getOrCreate()

import pandas as pd
import numpy as np
import seaborn as sns
import zipfile
from sklearn.model_selection import train_test_split
from catboost import CatBoostRegressor
from sklearn.pipeline import Pipeline
from sklearn.metrics import mean_absolute_error, mean_absolute_percentage_error, mean_squared_error
import matplotlib.pyplot as plt
from sklearn.model_selection import TimeSeriesSplit
from sklearn.preprocessing import OrdinalEncoder
from sklearn.compose import ColumnTransformer
from xgboost import XGBRegressor
import warnings
warnings.filterwarnings('ignore')

# Versões das bibliotecas principais (para reprodutibilidade)
import catboost, xgboost, sklearn
print(f"CatBoost: {catboost.__version__} | XGBoost: {xgboost.__version__} | scikit-learn: {sklearn.__version__}")

Importar base de dados


In [0]:
# Carregando a base gold do projeto pricing
order_data = spark.table("olist.gold.projeto_pricing").toPandas()
order_data.shape

separa teste e treino

In [0]:
order_data['order_purchase_timestamp'] = pd.to_datetime(order_data['order_purchase_timestamp'])
print(order_data['order_purchase_timestamp'].min(), order_data['order_purchase_timestamp'].max())

# volume de itens por mês
print(order_data.groupby(order_data['order_purchase_timestamp'].dt.to_period('M')).size())
# 3) novas colunas a partir da data
ts = order_data['order_purchase_timestamp']
order_data['purchase_month'] = ts.dt.month
order_data['purchase_dow'] = ts.dt.dayofweek               # 0 = segunda, 6 = domingo
order_data['purchase_week'] = ts.dt.isocalendar().week.astype(int)
order_data['purchase_is_weekend'] = (ts.dt.dayofweek >= 5).astype(int)
order_data = order_data.sort_values('order_purchase_timestamp').reset_index(drop=True)

X = order_data.drop(columns=['price', 'product_category_name_english']) #tirar pois ja tem em portugues
y = order_data['price']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, shuffle=False
)

print("Treino:", X_train.shape, "Teste:", X_test.shape)
print("Treino:", X_train['order_purchase_timestamp'].min(), "até", X_train['order_purchase_timestamp'].max())
print("Teste:", X_test['order_purchase_timestamp'].min(), "até", X_test['order_purchase_timestamp'].max())

Definição das variáveis preditoras

As variáveis categóricas originalmente armazenadas como texto são identificadas, assim como algumas variáveis numéricas que representam categorias  por exemplo, prefixo do CEP do vendedor e variáveis de calendário. Para o CatBoost, essas colunas categóricas são convertidas para strings.

A coluna `order_purchase_timestamp` não entra diretamente no conjunto de variáveis usado pelo modelo; suas informações de calendário são representadas pelas variáveis derivadas.

In [0]:
# extra_cat são as features categóricas disfarçadas de numéricas

extra_cat = ['purchase_month', 'purchase_dow', 'purchase_week']
cat_feat = X_train.select_dtypes(include='object').columns.tolist() + extra_cat
num_feat = [c for c in X_train.select_dtypes(include='number').columns if c not in extra_cat]


cat_feat_cb = cat_feat.copy()
num_feat_cb = num_feat.copy()

# Converter as variáveis categóricas para string
# Isso inclui categorias que originalmente são números
for col in cat_feat_cb:
    X_train[col] = X_train[col].astype(str)
    X_test[col] = X_test[col].astype(str)

print(f"Existem {len(cat_feat_cb)} variáveis categóricas: {cat_feat_cb}")
print(f"Existem {len(num_feat_cb)} variáveis numéricas: {num_feat_cb}")

##6. Treinamento dos modelos e ajuste de hiperparâmetros

A escolha inicial prioriza modelos de boosting baseados em árvores. A justificativa registrada no projeto é a presença de variáveis categóricas de alta cardinalidade e de relações potencialmente não lineares entre as características e o preço.

O CatBoost é o principal candidato por oferecer tratamento nativo para variáveis categóricas. Além da escala original do preço, será avaliada a transformação log no `price`. Essa transformação altera a escala do alvo durante o treinamento; as previsões do modelo transformado são convertidas de volta para reais antes da comparação das métricas.

Também serão testadas variações do conjunto de features: dimensões individuais (length, height, width) versus volume agregado, e com ou sem o valor do frete
 
Na escolha dos modelos para o treinamento, podemos elencar alguns melhores concorrentes. Entre as possibilidades estão modelos lineares, modelos de árvores e outras famílias, como KNN e redes neurais.

Como vimos no nosso EDA, nossa base de dados possui muita cardinalidade, e os modelos lineares não serão convenientes para isso: eles exigiriam transformar essas variáveis em milhares de colunas (one-hot) ou em encodings feitos à mão, o que deixa o modelo esparso e propenso a decorar categorias raras. Além disso, o EDA mostrou que a relação entre peso e preço não é uma reta (o preço quase estaciona entre 350 g e 1 kg), que o preço varia muito dentro de cada categoria e que peso, volume e dimensões são redundantes. Os modelos baseados em árvores lidam bem com essas características, sem exigir escala ou transformações nas variáveis numéricas, como o log que propusemos na etapa do EDA. O log do alvo (log1p(price)) é uma decisão à parte e continuará sendo testado. KNN e redes neurais também ficam de fora, por costumarem render menos em dados tabulares e darem mais trabalho.

Por isso, vamos priorizar modelos de boosting de árvores:

CatBoost: será o principal candidato, por tratar nativamente categóricas de alta cardinalidade, com uma técnica que reduz o vazamento do alvo.
XGBoost: entra como comparação, por ser o padrão do mercado em dados tabulares, mas costuma exigir mais cuidado com as categóricas de alta cardinalidade.


### Validação cruzada com hiperparâmetros padrão

Antes de ajustar hiperparâmetros, comparamos os modelos com configurações padrão para entender quais combinações de features e modelo apresentam as melhores métricas. Essa etapa serve como linha de base.

Usamos validação cruzada temporal (`TimeSeriesSplit` com 3 folds), que respeita a ordem cronológica dos dados: cada fold treina com os dados anteriores e valida com os posteriores, evitando vazamento de informações do futuro.

Testamos 16 combinações: 4 conjuntos de features (dimensões vs volume, com vs sem frete) × 2 modelos (CatBoost e XGBoost) × 2 versões do alvo (preço original e `log1p(price)`). Para cada combinação, tiramos a média das métricas dos 3 folds e ranqueamos por MAE de validação.

In [0]:
# Imports adicionais para validação cruzada
from sklearn.model_selection import TimeSeriesSplit
from sklearn.preprocessing import OrdinalEncoder
from sklearn.compose import ColumnTransformer
from xgboost import XGBRegressor
import warnings
warnings.filterwarnings('ignore')

# Criar feature de volume (sugerido pela EDA: correlação 0.40 com preço, resume as 3 dimensões)
for df in [X_train, X_test]:
    df['product_volume_cm3'] = (
        df['product_length_cm'] * df['product_height_cm'] * df['product_width_cm']
    )
print(f"X_train: {X_train.shape} | X_test: {X_test.shape}")

# ── Definição dos conjuntos de features para teste ─
# base: categóricas e numéricas que sempre entram
base_cat = ['seller_id', 'product_id', 'product_category_name',
            'seller_zip_code_prefix', 'seller_city', 'seller_state',
            'purchase_month', 'purchase_dow', 'purchase_week']
base_num = ['product_name_length', 'product_description_length',
            'product_photos_qty', 'product_weight_g', 'purchase_is_weekend']
dims_cols = ['product_length_cm', 'product_height_cm', 'product_width_cm']
volume_col = ['product_volume_cm3']
freight_col = ['freight_value']

# 4 feature sets: {volume, dims} × {com_frete, sem_frete}
# both (volume + dims) foi removido: volume = length × height × width, sao correlacionados
feature_sets = {
    'volume_sem_frete':  {'cat': base_cat, 'num': base_num + volume_col},
    'volume_com_frete':  {'cat': base_cat, 'num': base_num + volume_col + freight_col},
    'dims_sem_frete':    {'cat': base_cat, 'num': base_num + dims_cols},
    'dims_com_frete':    {'cat': base_cat, 'num': base_num + dims_cols + freight_col},
}

def evaluate_cv(model_type, fs, use_log, n_splits=3):
    """Roda TimeSeriesSplit CV e retorna métricas de treino E validação por fold.

    model_type: 'catboost' ou 'xgboost'
    fs: dict com 'cat' (lista de colunas categóricas) e 'num' (lista de colunas numéricas)
    use_log: True para log1p(y) no treino e expm1 na predição
    """
    cat_cols = fs['cat']
    num_cols = fs['num']
    all_cols = cat_cols + num_cols

    X = X_train[all_cols].copy()
    for col in cat_cols:
        X[col] = X[col].astype(str)

    tscv = TimeSeriesSplit(n_splits=n_splits)
    fold_results = []

    for fold, (tr_idx, val_idx) in enumerate(tscv.split(X)):
        X_tr, X_val = X.iloc[tr_idx], X.iloc[val_idx]
        y_tr, y_val = y_train.iloc[tr_idx], y_train.iloc[val_idx]

        # Criar modelo fresco a cada fold
        if model_type == 'catboost':
            model = CatBoostRegressor(
                iterations=300, learning_rate=0.1, depth=6,
                cat_features=cat_cols, verbose=0, random_seed=42
            )
        else:  # xgboost
            pre = ColumnTransformer([
                ('cat', OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1), cat_cols),
                ('num', 'passthrough', num_cols)
            ])
            model = Pipeline([
                ('pre', pre),
                ('model', XGBRegressor(n_estimators=300, learning_rate=0.1, max_depth=6,
                                       random_state=42, verbosity=0))
            ])

        # Transformar target se log
        y_tr_fit = np.log1p(y_tr) if use_log else y_tr
        model.fit(X_tr, y_tr_fit)

        # Predição (desfaz o log se necessário)
        if use_log:
            pred_tr = np.expm1(model.predict(X_tr))
            pred_val = np.expm1(model.predict(X_val))
        else:
            pred_tr = model.predict(X_tr)
            pred_val = model.predict(X_val)

        fold_results.append({
            'fold': fold + 1,
            'train_mae': mean_absolute_error(y_tr, pred_tr),
            'val_mae': mean_absolute_error(y_val, pred_val),
            'train_mape': mean_absolute_percentage_error(y_tr, pred_tr),
            'val_mape': mean_absolute_percentage_error(y_val, pred_val),
            'train_rmse': float(np.sqrt(mean_squared_error(y_tr, pred_tr))),
            'val_rmse': float(np.sqrt(mean_squared_error(y_val, pred_val))),
        })

    return fold_results

print(f"Função pronta. {len(feature_sets)} feature sets definidos.")

In [0]:
# Rodar todas as combinações: 4 feature_sets × 2 modelos × 2 log = 16 runs × 3 folds = 48 fits
all_results = []

for fs_name, fs in feature_sets.items():
    for model_type in ['catboost', 'xgboost']:
        for use_log in [False, True]:
            print(f"Testando: {fs_name} | {model_type} | log={use_log}...", end=" ")
            folds = evaluate_cv(model_type, fs, use_log, n_splits=3)
            for r in folds:
                r['feature_set'] = fs_name
                r['model'] = model_type
                r['log'] = use_log
                all_results.append(r)
            val_mae = np.mean([r['val_mae'] for r in folds])
            print(f"val_MAE={val_mae:.2f}")

# Resumo: média dos folds por combinação
results_df = pd.DataFrame(all_results)
summary = results_df.groupby(['feature_set', 'model', 'log']).agg({
    'train_mae': 'mean', 'val_mae': 'mean',
    'train_mape': 'mean', 'val_mape': 'mean',
    'train_rmse': 'mean', 'val_rmse': 'mean',
}).round(2)

# Gaps de todas as métricas (val - treino): indica overfitting
summary['gap_mae'] = (summary['val_mae'] - summary['train_mae']).round(2)
summary['gap_mape'] = (summary['val_mape'] - summary['train_mape']).round(2)
summary['gap_rmse'] = (summary['val_rmse'] - summary['train_rmse']).round(2)

# Reordenar colunas: agrupar por métrica (treino, val, gap)
summary = summary[['train_mae', 'val_mae', 'gap_mae',
                    'train_mape', 'val_mape', 'gap_mape',
                    'train_rmse', 'val_rmse', 'gap_rmse']]
summary = summary.sort_values('val_mae')
display(summary)

RMSE: quando erros grandes são muito mais custosos que vários pequenos

MAE: quando o custo do erro é proporcional ao seu tamanho

MAPE/WAPE: quando o erro relativo importa ou os valores têm escalas muito diferentes.

Neste projeto, priorizamos minimizar o MAE. Essa escolha pode variar conforme a estratégia do negócio. Os melhores modelos nessa avaliação foram:

In [0]:
# Top 5 combinações ordenadas por val MAE
top5 = summary.head(5).copy().reset_index()
top5['log'] = top5['log'].map({True: 'sim', False: 'nao'})
top5.columns = ['Feature Set', 'Modelo', 'Log',
                'MAE Treino', 'MAE Val', 'Gap MAE',
                'MAPE Treino', 'MAPE Val', 'Gap MAPE',
                'RMSE Treino', 'RMSE Val', 'Gap RMSE']
display(top5)

## Hiperparâmetro Tuning

Com a validação cruzada de hiperparâmetros padrão, identificamos os 2 melhores candidatos: **CatBoost** e **XGBoost**, ambos com o feature set `dims_com_frete` + `log1p(price)`. Agora ajustamos os hiperparâmetros de cada um para tentar melhorar o desempenho.

O ajuste é feito com busca aleatória: definimos um grid de valores para cada hiperparâmetro, amostramos 12 combinações aleatoriamente e avaliamos cada uma com validação cruzada temporal (`TimeSeriesSplit` com 3 folds). A combinação com o melhor MAE de validação é selecionada.

Para o CatBoost, testamos `iterations`, `learning_rate`, `depth` e `l2_leaf_reg`. Para o XGBoost, testamos `n_estimators`, `learning_rate`, `max_depth`, `reg_alpha`, `reg_lambda`, `subsample` e `colsample_bytree`.

Usamos um loop manual em vez do `RandomizedSearchCV` do scikit-learn porque ele não consegue clonar o parâmetro `cat_features` do CatBoost. O `X_test` não é tocado em nenhuma etapa do tuning — só será usado na avaliação final.

In [0]:
import random
from itertools import product

# Preparar dados (dims_com_frete + log, mesmo setup do CV)
fs = feature_sets['dims_com_frete']
cat_cols = fs['cat']
num_cols = fs['num']
all_cols = cat_cols + num_cols

X_tune = X_train[all_cols].copy()
for col in cat_cols:
    X_tune[col] = X_tune[col].astype(str)

tscv = TimeSeriesSplit(n_splits=3)

# CatBoost: loop manual (RandomizedSearchCV nao consegue clonar cat_features)
# Grid ajustado para reduzir overfitting:
# - depth max 8 (removido 10)
# - learning_rate max 0.1 (removido 0.15, 0.2; adicionado 0.03)
# - l2_leaf_reg ate 20 (mais regularizacao)
# - rsm: subamostragem de colunas a cada split
# - mais trials (15) para explorar melhor
random.seed(42)
cat_param_grid = {
    'iterations': [300, 500, 800],
    'learning_rate': [0.01, 0.03, 0.05, 0.1],
    'depth': [4, 6, 8],
    'l2_leaf_reg': [3, 5, 7, 10, 15, 20],
    'rsm': [0.7, 0.8, 0.9, 1.0],
}

all_combos = list(product(*cat_param_grid.values()))
sampled_combos = random.sample(all_combos, min(15, len(all_combos)))

best_val_mae = float('inf')
best_params = None
best_train_mae = None

for i, combo in enumerate(sampled_combos):
    params = dict(zip(cat_param_grid.keys(), combo))
    print(f"  Trial {i+1}/15: {params}...", end=" ")
    
    fold_train, fold_val = [], []
    for tr_idx, val_idx in tscv.split(X_tune):
        X_tr, X_val = X_tune.iloc[tr_idx], X_tune.iloc[val_idx]
        y_tr, y_val = y_train.iloc[tr_idx], y_train.iloc[val_idx]
        
        model = CatBoostRegressor(**params, cat_features=cat_cols, verbose=0, random_seed=42)
        model.fit(X_tr, np.log1p(y_tr))
        
        pred_tr = np.expm1(model.predict(X_tr))
        pred_val = np.expm1(model.predict(X_val))
        fold_train.append(mean_absolute_error(y_tr, pred_tr))
        fold_val.append(mean_absolute_error(y_val, pred_val))
    
    train_mae = np.mean(fold_train)
    val_mae = np.mean(fold_val)
    print(f"val MAE={val_mae:.2f}")
    
    if val_mae < best_val_mae:
        best_val_mae = val_mae
        best_params = params
        best_train_mae = train_mae

print(f"\n=== CatBoost Tuning ===")
print(f"Melhores params: {best_params}")
print(f"Train MAE: {best_train_mae:.2f} | Val MAE: {best_val_mae:.2f} | Gap: {best_val_mae - best_train_mae:.2f}")
print(f"Sem tuning val MAE era 47.65 -> depois: {best_val_mae:.2f}")

cat_best_val = best_val_mae
cat_best_params = best_params

In [0]:
# XGBoost: loop manual (mesma abordagem do CatBoost)
xgb_param_grid = {
    'n_estimators': [200, 300, 500],
    'learning_rate': [0.01, 0.05, 0.1, 0.15, 0.2],
    'max_depth': [4, 6, 8, 10],
    'reg_alpha': [0, 0.1, 1, 10],
    'reg_lambda': [1, 5, 10, 20],
    'subsample': [0.7, 0.8, 0.9, 1.0],
    'colsample_bytree': [0.7, 0.8, 0.9, 1.0],
}

random.seed(42)
all_combos = list(product(*xgb_param_grid.values()))
sampled_combos = random.sample(all_combos, min(12, len(all_combos)))

best_val_mae = float('inf')
best_params = None
best_train_mae = None

for i, combo in enumerate(sampled_combos):
    params = dict(zip(xgb_param_grid.keys(), combo))
    print(f"  Trial {i+1}/12: {params}...", end=" ")
    
    fold_train, fold_val = [], []
    for tr_idx, val_idx in tscv.split(X_tune):
        X_tr, X_val = X_tune.iloc[tr_idx], X_tune.iloc[val_idx]
        y_tr, y_val = y_train.iloc[tr_idx], y_train.iloc[val_idx]
        
        model = Pipeline([
            ('pre', ColumnTransformer([
                ('cat', OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1), cat_cols),
                ('num', 'passthrough', num_cols)
            ])),
            ('model', XGBRegressor(**params, random_state=42, verbosity=0))
        ])
        model.fit(X_tr, np.log1p(y_tr))
        
        pred_tr = np.expm1(model.predict(X_tr))
        pred_val = np.expm1(model.predict(X_val))
        fold_train.append(mean_absolute_error(y_tr, pred_tr))
        fold_val.append(mean_absolute_error(y_val, pred_val))
    
    train_mae = np.mean(fold_train)
    val_mae = np.mean(fold_val)
    print(f"val MAE={val_mae:.2f}")
    
    if val_mae < best_val_mae:
        best_val_mae = val_mae
        best_params = params
        best_train_mae = train_mae

print(f"\n=== XGBoost Tuning ===")
print(f"Melhores params: {best_params}")
print(f"Train MAE: {best_train_mae:.2f} | Val MAE: {best_val_mae:.2f} | Gap: {best_val_mae - best_train_mae:.2f}")
print(f"Sem tuning val MAE era 49.72 -> depois: {best_val_mae:.2f}")

xgb_best_val = best_val_mae
xgb_best_params = best_params

In [0]:
print("=== Comparação Pós-Tuning ===\n")

print(f"CatBoost:  val MAE = {cat_best_val:.2f} (era 47.65)")
print(f"XGBoost:   val MAE = {xgb_best_val:.2f} (era 49.72)")

melhor = "CatBoost" if cat_best_val < xgb_best_val else "XGBoost"
print(f"\nMelhor modelo: {melhor}")
print(f"Diferença: {abs(cat_best_val - xgb_best_val):.2f} MAE")

## 7. Teste e avaliação final do modelo

Após o tuning, CatBoost e XGBoost ficaram praticamente empatados (val MAE de 44,54 vs 44,44 — diferença de 0,10). O CatBoost foi mantido para a avaliação final por ter o pipeline com `cat_features` nativo. Agora treinamos o modelo final com os melhores hiperparâmetros no **treino completo** (`X_train` inteiro, sem split de validação) e avaliamos no **`X_test`**, que nunca foi usado em nenhuma etapa anterior. Isso dá a estimativa mais honesta do erro em produção.

Usamos o mesmo setup do tuning: feature set `dims_com_frete`, transformação `log1p(price)` e conversão das categóricas para string.

### Análise dos resultados

O MAE de teste (~R$ 49) ainda é alto em relação à mediana do preço (R$ 74,90 vista na EDA), o que significa um erro médio de aproximadamente 66% do preço mediano. Esse erro é difícil de reduzir apenas com ajuste de hiperparâmetros.

Tentamos reduzir o overfitting ajustando o grid de tuning (removendo `depth=10`, diminuindo o `learning_rate` máximo, adicionando `rsm` para subamostragem de colunas e aumentando `l2_leaf_reg`). O tuning reduziu o val MAE de 47,65 para 44,54, mas o test MAE ficou em 49,38 — o gap entre validação e teste indica que o modelo ainda não generaliza completamente para produtos e períodos novos. Isso é um trade-off: modelos mais regularizados generalizam melhor mas perdem capacidade de capturar padrões.



In [0]:
# Treinar modelo final com os melhores hiperparâmetros no treino completo
fs = feature_sets['dims_com_frete']
cat_cols = fs['cat']
num_cols = fs['num']
all_cols = cat_cols + num_cols

X_tr_final = X_train[all_cols].copy()
X_te_final = X_test[all_cols].copy()

for col in cat_cols:
    X_tr_final[col] = X_tr_final[col].astype(str)
    X_te_final[col] = X_te_final[col].astype(str)

# CatBoost com os melhores params do tuning
final_model = CatBoostRegressor(**cat_best_params, cat_features=cat_cols, verbose=0, random_seed=42)
final_model.fit(X_tr_final, np.log1p(y_train))

# Predição no teste (desfaz o log)
pred_test = np.expm1(final_model.predict(X_te_final))
pred_train = np.expm1(final_model.predict(X_tr_final))

# Métricas no teste
test_mae = mean_absolute_error(y_test, pred_test)
test_mape = mean_absolute_percentage_error(y_test, pred_test)
test_rmse = float(np.sqrt(mean_squared_error(y_test, pred_test)))

# Métricas no treino (para comparar gap)
train_mae = mean_absolute_error(y_train, pred_train)
train_mape = mean_absolute_percentage_error(y_train, pred_train)
train_rmse = float(np.sqrt(mean_squared_error(y_train, pred_train)))

print("=== Avaliação Final — CatBoost (dims_com_frete + log) ===")
print(f"\nHiperparâmetros: {cat_best_params}")
print(f"\n{'Métrica':<10} {'Treino':>10} {'Teste':>10} {'Gap':>10}")
print(f"{'MAE':<10} {train_mae:>10.2f} {test_mae:>10.2f} {test_mae - train_mae:>10.2f}")
print(f"{'MAPE':<10} {train_mape:>10.4f} {test_mape:>10.4f} {test_mape - train_mape:>10.4f}")
print(f"{'RMSE':<10} {train_rmse:>10.2f} {test_rmse:>10.2f} {test_rmse - train_rmse:>10.2f}")
print(f"\nVal MAE (tuning): {cat_best_val:.2f} -> Test MAE: {test_mae:.2f}")

# Amostra de previsões vs real
amostra = pd.DataFrame({'real': y_test.values[:10], 'previsto': pred_test[:10]})
print("\nAmostra (10 primeiros):")
display(amostra.round(2))

### Análise do overfitting e experimento sem `product_id` e `seller_id`

 

O gap entre treino e teste no modelo final (MAE de 18,92) indica **overfitting**: o modelo está decorando os dados de treino em vez de generalizar. As principais suspeitas são as variáveis de alta cardinalidade:

- **`product_id`**: 26.078 produtos únicos, e **44% das linhas do teste são de produtos que o treino nunca viu**. O modelo aprende um preço específico para cada produto no treino, mas não consegue prever produtos desconhecidos.
- **`seller_id`**: 2.477 vendedores, 995 com menos de 5 vendas. Vendedores raros no treino podem não aparecer no teste, e o modelo não generaliza para eles.

Essas features permitem que o modelo "memorize" o preço de cada produto/vendedor individual no treino (por isso o train MAE é baixo), mas essa memorização não transfere para o teste, onde quase metade dos produtos é nova.

**Experimento:** vamos treinar o mesmo modelo CatBoost (`dims_com_frete` + `log1p`), com os mesmos hiperparâmetros do tuning, mas **removendo `product_id` e `seller_id`**. O objetivo é verificar se o gap treino-teste diminui e se o modelo generaliza melhor, mesmo que o MAE absoluto possa mudar.

In [0]:
# Modelo sem product_id e seller_id: mesmo setup do modelo final, removendo as 2 variáveis de alta cardinalidade
fs = feature_sets['dims_com_frete']
cat_cols = [c for c in fs['cat'] if c not in ['product_id', 'seller_id']]  # remove as 2 variáveis
num_cols = fs['num']
all_cols = cat_cols + num_cols

print(f"Features removidas: product_id, seller_id")
print(f"Categóricas ({len(cat_cols)}): {cat_cols}")
print(f"Numéricas ({len(num_cols)}): {num_cols}")

X_tr_final = X_train[all_cols].copy()
X_te_final = X_test[all_cols].copy()

for col in cat_cols:
    X_tr_final[col] = X_tr_final[col].astype(str)
    X_te_final[col] = X_te_final[col].astype(str)

# Usar os melhores hiperparâmetros do tuning anterior (com fallback caso não esteja definido)
best_params = cat_best_params if 'cat_best_params' in globals() else {
    'iterations': 500, 'learning_rate': 0.1, 'depth': 8, 'l2_leaf_reg': 5, 'rsm': 0.9
}

# CatBoost sem product_id e seller_id
final_model_no_ids = CatBoostRegressor(**best_params, cat_features=cat_cols, verbose=0, random_seed=42)
final_model_no_ids.fit(X_tr_final, np.log1p(y_train))

# Predição no teste (desfaz o log)
pred_test = np.expm1(final_model_no_ids.predict(X_te_final))
pred_train = np.expm1(final_model_no_ids.predict(X_tr_final))

# Métricas no teste
test_mae = mean_absolute_error(y_test, pred_test)
test_mape = mean_absolute_percentage_error(y_test, pred_test)
test_rmse = float(np.sqrt(mean_squared_error(y_test, pred_test)))

# Métricas no treino (para comparar gap)
train_mae = mean_absolute_error(y_train, pred_train)
train_mape = mean_absolute_percentage_error(y_train, pred_train)
train_rmse = float(np.sqrt(mean_squared_error(y_train, pred_train)))

print(f"\n=== Avaliação — CatBoost SEM product_id e seller_id (dims_com_frete + log) ===")
print(f"\nHiperparâmetros: {best_params}")
print(f"\n{'Métrica':<10} {'Treino':>10} {'Teste':>10} {'Gap':>10}")
print(f"{'MAE':<10} {train_mae:>10.2f} {test_mae:>10.2f} {test_mae - train_mae:>10.2f}")
print(f"{'MAPE':<10} {train_mape:>10.4f} {test_mape:>10.4f} {test_mape - train_mape:>10.4f}")
print(f"{'RMSE':<10} {train_rmse:>10.2f} {test_rmse:>10.2f} {test_rmse - train_rmse:>10.2f}")
print(f"\nModelo anterior (com IDs): Train MAE=30.47 | Test MAE=49.38 | Gap=18.92")
print(f"Modelo sem IDs:            Train MAE={train_mae:.2f} | Test MAE={test_mae:.2f} | Gap={test_mae - train_mae:.2f}")

# Amostra de previsões vs real
amostra = pd.DataFrame({'real': y_test.values[:10], 'previsto': pred_test[:10]})
print("\nAmostra (10 primeiros):")
display(amostra.round(2))

## 8. Interpretação dos resultados e insights de negócio

### Resumo do desempenho

O melhor modelo (CatBoost com `dims_com_frete` + `log1p`) alcançou **MAE de R$ 49,38** no teste, contra R$ 82,10 de um baseline que prevê a mediana. Isso representa uma **melhoria de ~40%** e um **R² de aproximadamente 0,47** — o modelo explica cerca de metade da variância dos preços.

| Métrica | Treino | Teste | Significado |
| --- | --- | --- | --- |
| MAE | 30,47 | 49,38 | Erro médio de ~R$ 49 por produto |
| MAPE | 0,22 | 0,46 | Erro médio de ~46% em relação ao preço real |
| RMSE | 106,47 | 147,35 | Erros grandes pontuais (RMSE 3x o MAE) |

### Por que o modelo está próximo do teto

A EDA mostrou que **nenhuma variável isolada explica bem o preço**:

- A correlação mais forte é o peso (Spearman 0,53) — longe de 1,0
- Frete (0,43) e volume (0,40) são moderadas
- Nome e fotos quase não têm relação (0,04 e 0,02)
- Dentro de cada categoria, o preço varia de 2 a 4x (IQR)

Os outros ~53% da variância vêm de fatores que **não estão na base**: marca, concorrência entre vendedores, estratégia de margem, reviews, sazonalidade específica e percepção de valor do cliente. Não há feature que se possa construir a partir dos dados disponíveis que capture esses fatores.

### O que tentamos e por que não melhorou

| Experimento | Resultado | Por quê |
| --- | --- | --- |
| Remover `product_id` e `seller_id` | Gap caiu 18,92 → 18,38, mas test MAE subiu 49,38 → 50,98 | Os IDs causam overfitting, mas também carregam sinal real para os 56% dos produtos conhecidos no teste |
| Tuning de hiperparâmetros (CatBoost e XGBoost) | Val MAE 47,65 → 44,54 | Melhoria real, mas o test MAE continuou em ~49 — o gap val→teste indica que o limite é do dado, não do modelo |
| 4 feature sets (volume vs dimensões × com/sem frete) | Diferença máxima de ~1 MAE entre eles | Volume e dimensões são redundantes (correlação 0,74-0,77); o frete ajuda pouco e é questionável em produção |
| Log vs sem log | Log melhorou ~8 MAE no CatBoost | O log comprime a cauda e ajuda, mas é uma transformação, não nova informação |

**Padrão observado:** todas as intervenções (tuning, feature sets, remoção de IDs) mudam o MAE em **±1 a 3 reais**. Nenhuma alteração trouxe um salto significativo, porque o gargalo não está no modelo — está na **ausência de features preditivas**.

### Limitação estrutural: produtos desconhecidos

**44% das linhas do teste são de produtos que o treino nunca viu.** Isso significa que quase metade das previsões precisa ser feita só com base em categoria, peso, dimensões e localização do vendedor — sem nenhum histórico do produto. Para esses itens, o modelo essencialmente prevê a mediana da categoria, e o erro é maior.

### Insight de negócio: o frete como variável

O feature set com melhor desempenho inclui `freight_value`. Porém, o frete depende do destino do cliente e só é conhecido **depois** que o pedido é feito. Se o objetivo do modelo é precificar produtos **antes** da compra (ex: sugerir preço para o vendedor), o frete não estará disponível e o modelo com `dims_sem_frete` é o correto para produção, mesmo com MAE levemente pior.

### Conclusão final

O modelo está **próximo do teto do que os dados permitem**. O pipeline está correto: split temporal, validação cruzada temporal, transformação log, tuning, isolamento do teste. As métricas (~R$ 49 de erro médio, 46% de MAPE) refletem a **dificuldade inerente de prever preço a partir de características físicas e logísticas** — não uma deficiência do modelo.

Para melhorar significativamente, seria necessário enriquecer a base com:
- **Conteúdo textual** da descrição do produto (não apenas o tamanho)
- **Reviews e notas** dos produtos e vendedores
- **Histórico de preços** do mesmo produto ao longo do tempo
- **Marca** do produto

Esses dados estão fora do escopo da base Olist disponível.